# Panorama Image Stitching
## Midterm Report — Proposal and Sequential Baseline

**CSC14120 · Parallel Programming · Group 16 · V0**

Notebook trình bày pipeline ghép hai ảnh, kiểm chứng phiên bản C++ tuần tự và chuẩn bị kế hoạch CUDA. Các bảng sai số, timing và hình ảnh kết quả được sinh từ lần chạy hiện tại. Các tham số thiết kế được đối chiếu với mã nguồn đã nộp.

| Thông tin | Nội dung |
|:--|:--|
| Người tổ chức notebook | Thanh Chuong — 23120195 |
| Mốc bài nộp | 23/10/2026 |
| Môi trường chính thức | Google Colab · Tesla T4 · `nvcc -arch=sm_75` |
| Phiên bản thuật toán | V0 tuần tự; chưa triển khai CUDA trong notebook này |
| Trạng thái P3-T09 | Khung thực thi; benchmark đang chờ script P3-T04 của Thành |

> Bản Git không lưu output. Bản nộp cuối phải chạy **Runtime → Run all** trên phiên Colab T4 mới và lưu toàn bộ output. Không sử dụng các số đo ngày trước thay cho output của lần chạy này.

### Contents

- [0. Reproducibility Setup](#setup)
- [3.1 Team Information and Work Plan](#team)
- [3.2 Problem Understanding](#problem)
- [3.3 Chosen Methods](#methods)
- [3.4 Sequential Implementation](#implementation)
- [3.5 Results and Profiling](#results)
- [3.6 Dataset](#dataset)
- [3.7 GPU Execution Plan](#gpu-plan)
- [References](#references)
- [Reproducibility Checklist](#checklist)

<a id="setup"></a>
## 0. Reproducibility Setup
### 0.1 Runtime Verification

Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy. V0 thực thi trên CPU của phiên Colab; T4 là môi trường build/test theo quy định môn học. Tên CPU được lưu để diễn giải kết quả V0.

In [ ]:
# Authors: Thanh Chuong (23120195)
# External source: none
from pathlib import Path
import csv
import datetime as dt
import hashlib
import io
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import uuid

def execute(command, cwd=None, timeout=1800, check=True):
    """Run one explicit command; a failure is reported without an automatic retry."""
    print("$", shlex.join([str(part) for part in command]), flush=True)
    result = subprocess.run([str(part) for part in command], cwd=cwd,
                            capture_output=True, text=True, timeout=timeout)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check:
        result.check_returncode()
    return result

gpu_output = execute(["nvidia-smi"]).stdout
gpu_name = execute(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip()
if not re.search(r"\bT4\b", gpu_name):
    raise RuntimeError("Official execution requires a Tesla T4 Colab runtime.")
compiler_version = execute(["nvcc", "--version"]).stdout
cpu_lines = Path("/proc/cpuinfo").read_text().splitlines()
cpu_name = next((line.split(":", 1)[1].strip() for line in cpu_lines
                 if line.startswith("model name")), platform.processor())
SESSION_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
ENVIRONMENT = {"session_id": SESSION_ID, "started_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
               "gpu": gpu_name, "cpu": cpu_name, "python": sys.version,
               "platform": platform.platform(), "nvcc": compiler_version}
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))

### 0.2 Input Configuration

Chỉ sửa cell cấu hình dưới đây trước khi Run all. Chuẩn bị một thư mục workspace hoặc ZIP có `src/`, `scripts/`, `tests/`, `docs/`, `dataset/` và `raw/tuan/standard1_left.jpg`.

- **`upload`**: tải một ZIP workspace lên Colab khi được yêu cầu; ZIP có thể có một thư mục gốc bao ngoài.
- **`drive`**: mount Drive, đặt `DRIVE_WORKSPACE` đến thư mục chứa trực tiếp các thư mục trên; không lưu token trong notebook.
- **`existing`**: dùng workspace đã có tại `EXISTING_WORKSPACE`.

Dataset chuẩn bị: [Google Drive](https://drive.google.com/drive/folders/1HOvnly1k1QHOsV17qLzSAG4ihszNzCqO?usp=sharing). Ảnh gốc: [Google Drive](https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing).

Script benchmark của Thành chưa có trong repo tại lúc soạn. `BENCHMARK_ARGS=None` giữ phần đó ở trạng thái chờ. Khi script đã có, điền **đúng các đối số do Thành cung cấp**; không coi giao diện đề xuất bên dưới là giao diện đã được xác nhận.

In [ ]:
SOURCE_MODE = "upload"  # upload, drive, or existing
SOURCE_ARCHIVE = Path("/content/Group16_workspace.zip")
DRIVE_WORKSPACE = "/content/drive/MyDrive/Group16/workspace"
EXISTING_WORKSPACE = "/content/panorama-stitching"
RAW_SYNTHETIC_RELATIVE = "raw/tuan/standard1_left.jpg"
EXPECTED_SYNTHETIC_SHA256 = "e9fbe7ecfed6344a36610fe50f2c6019bcd1525f4e217c033ed8ded4f90bf1e8"
PANORAMA_SIZE = "medium"
RUNS = 5
BENCHMARK_ARGS = None  # Replace with Thanh's confirmed argument list, not a shell string.
ACCELERATED_STAGES = []  # Fill with the team's P3-T01 decision, e.g. [1, 2, 3, 4, 5, 6].
QUALITY_REVIEW_CONFIRMED = False  # Set only after inspecting panoramas and native-resolution crops.
FINAL_REPORT_CONFIRMED = False  # Set after P4 merges and checks the team's report sections.
REQUIRE_COMPLETE_RUN = False  # True for the final submission; pending evidence then raises an error.

In [ ]:
# Reject unsafe archive paths before extracting any member.
import stat
import zipfile

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path(DRIVE_WORKSPACE).resolve()
elif SOURCE_MODE == "existing":
    PROJECT = Path(EXISTING_WORKSPACE).resolve()
elif SOURCE_MODE == "upload":
    if not SOURCE_ARCHIVE.is_file():
        from google.colab import files
        print("Tải lên một ZIP workspace gồm nguồn, tests, scripts, dataset và ảnh synthetic gốc.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one workspace ZIP archive.")
        SOURCE_ARCHIVE = Path(next(iter(uploaded))).resolve()
        del uploaded
    staging = Path("/content") / ("group16-source-" + SESSION_ID)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
        targets = set()
        for entry in bundle.infolist():
            target = (staging / entry.filename.replace("\\", "/")).resolve()
            if (not target.is_relative_to(staging) or target == staging or target in targets
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe or duplicate archive member: " + entry.filename)
            targets.add(target)
        for entry in bundle.infolist():
            target = staging / entry.filename.replace("\\", "/")
            if entry.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(entry) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
    roots = sorted({path.parent for path in staging.rglob("src")
                    if path.is_dir() and (path / "main.cu").is_file()})
    if len(roots) != 1:
        raise ValueError("The archive must contain exactly one project src/main.cu.")
    PROJECT = roots[0]
else:
    raise ValueError("SOURCE_MODE must be upload, drive, or existing.")

required = ["src/main.cu", "scripts/midterm_notebook.py", "tests/run_v0_features.py",
            "scripts/make_synthetic_pair.py", "scripts/homography_error.py", "dataset/manifest.csv",
            RAW_SYNTHETIC_RELATIVE]
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise FileNotFoundError("Missing workspace inputs: " + ", ".join(missing))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
RESULTS = PROJECT / "results" / "midterm" / SESSION_ID
BUILD = PROJECT / "build" / "midterm"
RESULTS.mkdir(parents=True, exist_ok=False)
BUILD.mkdir(parents=True, exist_ok=True)
print("Workspace:", PROJECT)
print("Current-session evidence:", RESULTS)

### 0.3 Dependencies and Provenance

Python chỉ phục vụ chuẩn bị dữ liệu, kiểm chứng và trình bày; chương trình `src/` vẫn chỉ dùng C/C++ chuẩn và stb. Mỗi phiên lưu hash nguồn và phiên bản Python packages. Dataset và kết quả không được đưa vào Git.

In [ ]:
import importlib.metadata as package_metadata
from PIL import Image as PILImage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as DisplayImage
from midterm_notebook import (SIZES, SITES, TIMING_COLUMNS, sha256, validate_dataset,
                              parse_stitch_csv, collect_benchmark_csv,
                              source_manifest, preview_image, overlap_box)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 10})
ENVIRONMENT["packages"] = {name: package_metadata.version(name)
                           for name in ("numpy", "Pillow", "pandas", "matplotlib")}
SOURCE_HASHES = source_manifest(PROJECT)
git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT, capture_output=True, text=True)
ENVIRONMENT["git_commit"] = git.stdout.strip() if git.returncode == 0 else "unavailable (uploaded source)"
(RESULTS / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
(RESULTS / "source_manifest.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")
print("Source files fingerprinted:", len(SOURCE_HASHES))
print(json.dumps(ENVIRONMENT["packages"], indent=2))
CHECKS = {"runtime_t4": True, "source_recorded": True}

### 0.4 Dataset Integrity

Kiểm tra đủ bốn cặp, ba kích thước, RGB JPEG, không EXIF và SHA-256 trùng manifest. Sai dữ liệu phải dừng trước build/đo; không bỏ âm thầm cặp lỗi để tạo bảng thiếu.

In [ ]:
DATASET = PROJECT / "dataset"
verified_images = validate_dataset(DATASET)
display(pd.DataFrame(verified_images)[["file", "width", "height", "sha256"]])
RAW_SYNTHETIC = PROJECT / RAW_SYNTHETIC_RELATIVE
raw_digest = sha256(RAW_SYNTHETIC)
print("Synthetic source:", RAW_SYNTHETIC_RELATIVE, "SHA-256:", raw_digest)
if EXPECTED_SYNTHETIC_SHA256 and raw_digest != EXPECTED_SYNTHETIC_SHA256:
    raise ValueError("Synthetic source differs from the source documented in S5 section 5.5.")
CHECKS["dataset_verified"] = True
CHECKS["synthetic_source_verified"] = True

<a id="team"></a>
## 3.1 Team Information and Work Plan

Phân công theo kế hoạch nhóm. Tuấn đã thực hiện tích hợp P2 và bổ sung bằng chứng kiểm thử; tên ở đầu từng file phản ánh người thực sự viết. Trách nhiệm stage và tác giả implementation có thể khác nhau.

| Thành viên | MSSV | Stage phụ trách | V1 dự kiến | V2/V3 |
|:--|:--|:--|:--|:--|
| Tuấn | 23120185 | S1 — Preprocessing | Kernel S1 | Chốt sau profiling V1 |
| Thành | 23120166 | S2 — Detection; S3 — Description | Kernel S2/S3 | Chốt sau profiling V1 |
| Đế | 23120232 | S4 — Matching; S5 — Homography | Kernel S4/S5 | Chốt sau profiling V1 |
| Chương | 23120195 | S6 — Warping and Blending | Kernel S6 | Chốt sau profiling V1 |

P3-T04/T05 do Thành phụ trách benchmark; Chương phụ trách khung notebook P3-T09 và báo cáo V0/S6 P3-T10. Quyết định GPU cuối cùng thuộc P3-T01 của cả nhóm, dựa trên output profiling.

<a id="problem"></a>
## 3.2 Problem Understanding

Đầu vào là hai ảnh A (trái), B (phải) có vùng chồng lấn. Pipeline tìm các chi tiết tương ứng, ước lượng ma trận homography và lấy mẫu ảnh vào một canvas chung. Cảnh ít parallax và chuyển động quay máy phù hợp với mô hình một homography; chuyển động vật thể và chênh sáng có thể gây ảnh kép hoặc đường chuyển sáng.

$$\mathbf{x}_A \sim H_{B\rightarrow A}\mathbf{x}_B,\qquad
\mathbf{x}_B \sim H^{-1}_{B\rightarrow A}\mathbf{x}_A.$$

Tọa độ có gốc trái trên, x sang phải, y xuống dưới; tâm pixel ở tọa độ nguyên. H lưu row-major. Chiều **B → A** được giữ thống nhất trong generator, S5, S6 và phép đánh giá sai số.

**RGB A/B → S1 grayscale/blur → S2 FAST → S3 BRIEF → S4 Hamming → S5 RANSAC/H → S6 inverse warp/blend → panorama.**

S1–S3 xử lý cả hai ảnh. S4 ghép descriptor; S5 lọc correspondence hình học; S6 nội suy và composite. Nội suy hoặc blending không sửa được một H sai.

<a id="methods"></a>
## 3.3 Chosen Methods

### 3.3.1 Algorithm and Parameter Summary

Bảng dưới tóm tắt thiết kế đã cài; output debug ở mục 3.4 in cấu hình thực tế. Không thay đổi tham số riêng cho một phiên bản để tạo speedup.

| Stage | Phương pháp V0 | Tham số/quy ước chính | Đặc điểm khi chuyển GPU |
|:--|:--|:--|:--|
| S1 | RGB → grayscale, Gaussian separable | σ = 1, radius = 3 | Pixel độc lập trong từng pass; đọc lân cận |
| S2 | FAST-9, score V, NMS 3×3, top-N | threshold = 0.05; tối đa 2000 keypoint; tie-break tất định | Score/NMS song song; selection cần điều phối |
| S3 | BRIEF không orientation, 256 bit | Patch 31×31; blur σ = 2, radius = 4 | Các descriptor độc lập; không gọi đây là ORB đầy đủ |
| S4 | Exact 2-NN Hamming và ratio test | Ratio = 0.80; cross-check tắt | So sánh cặp độc lập, reduction tìm 2-NN |
| S5 | Uniform RANSAC, normalized DLT, MSAC, refit | Seed = 16; cap = 2000; tau scale theo kích thước | Hypothesis/residual song song; giữ quy tắc stopping |
| S6 | Inverse mapping, bilinear, average overlap | H B → A; canvas cap 100 triệu pixel; tối đa 10.000 mỗi chiều | Mỗi pixel đích một thread; gather từ ảnh B |

### 3.3.2 Warping and Blending Contract

S6 chiếu các góc B qua H, hợp với giới hạn A để tìm canvas. Bbox dùng floor/ceil theo quy tắc đã chốt, kích thước bao gồm hai đầu mút. Với mỗi pixel canvas, trừ offset về hệ A rồi dùng H⁻¹ tìm vị trí trong B. Bilinear lấy bốn pixel lân cận; ở overlap, bản hiện tại dùng trung bình 50/50. Màu đen vẫn là dữ liệu hợp lệ; validity lấy từ hình học, không lấy từ màu.

Các trường hợp H suy biến, horizon cắt miền B, canvas quá lớn và allocation failure trả mã lỗi. H và −H biểu diễn cùng phép chiếu, nên mẫu số âm trên toàn miền không tự động là lỗi.

Chi tiết và phương án đã cân nhắc nằm trong `docs/stages/s1-preprocessing.md` đến `s6-warp-blend.md`. Các bản nháp 3.3 của từng thành viên sẽ được ghép và rà soát ở P4.

<a id="implementation"></a>
## 3.4 Sequential Implementation

### 3.4.1 Build the Submitted Source

Chỉ build từ file trong `src/`; không tạo bản sao thuật toán trong notebook. Dùng C++11, `sm_75`, tắt vectorization tự động để kiểm soát baseline; không OpenMP, thread hay SIMD tự viết. Lệnh và cờ build được lưu cùng kết quả.

`V0Workspace` quản lý buffer giữa các lần chạy. S6 chuyển ownership panorama cho workspace. Đọc/ghi ảnh và giải phóng output cũ ở ngoài vùng đo; các allocation bên trong thuật toán vẫn thuộc thời gian stage tương ứng.

In [ ]:
CPP_SOURCES = sorted(str(path.relative_to(PROJECT)) for path in (PROJECT / "src").glob("*.cpp"))
NVCC_FLAGS = ["-arch=sm_75", "-std=c++11", "-O2", "--fmad=false",
              "-Xcompiler=-ffp-contract=off,-fno-tree-vectorize,-fno-tree-slp-vectorize"]
STITCH = BUILD / "stitch"
build_command = ["nvcc", *NVCC_FLAGS, "src/main.cu", *CPP_SOURCES, "-o", STITCH]
result = execute(build_command, cwd=PROJECT)
(RESULTS / "build.log").write_text(result.stdout + result.stderr, encoding="utf-8")
(RESULTS / "build_command.json").write_text(json.dumps([str(x) for x in build_command], indent=2))
CHECKS["build"] = True

### 3.4.2 Unit and Integration Tests

Chạy bộ test đã có cho S1–S6, cấu hình và pipeline. Bộ test C++ được link với module thật; không tạo ground truth bằng chính hàm đang kiểm thử. Các test Python kiểm tra công cụ dataset/synthetic và helper notebook. Test không thuộc benchmark; lỗi test dừng phiên chạy.

In [ ]:
test_logs = []
result = execute([sys.executable, "tests/run_v0_features.py", "--compiler", "nvcc", "--require-t4"], cwd=PROJECT)
test_logs.append(result.stdout + result.stderr)
# The existing runner covers S4, S5, allocation, and S6. Cover the remaining modules here.
EXTRA_TESTS = ["test_v0_preprocess", "test_v0_detect", "test_v0_describe",
               "test_v0_detect_describe", "test_v0_config", "test_v0_pipeline"]
for name in EXTRA_TESTS:
    binary = BUILD / name
    execute(["nvcc", *NVCC_FLAGS, "-Isrc", *CPP_SOURCES, f"tests/{name}.cpp", "-o", binary], cwd=PROJECT)
    result = execute([binary], cwd=PROJECT)
    if "PASS" not in result.stdout:
        raise RuntimeError("Test did not report PASS: " + name)
    test_logs.append(result.stdout + result.stderr)
for test_file in ("test_dataset_scripts.py", "test_synthetic_scripts.py", "test_midterm_notebook.py"):
    result = execute([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", test_file], cwd=PROJECT)
    test_logs.append(result.stdout + result.stderr)
(RESULTS / "tests.log").write_text("\n".join(test_logs), encoding="utf-8")
CHECKS["tests"] = True

### 3.4.3 Command-Line and Error Handling

CLI: `stitch <left> <right> <output> --version 0 [--runs N] [--debug]`. Mặc định N = 5. Log debug đi vào stderr, CSV đi vào stdout. Các ca dưới kiểm tra cờ, version, số lần chạy và file lỗi; không đo các ca thất bại.

In [ ]:
left = DATASET / "site1/small_left.jpg"
right = DATASET / "site1/small_right.jpg"
output = RESULTS / "cli_output.png"
bad_file = RESULTS / "not_an_image.jpg"
bad_file.write_text("invalid image bytes", encoding="utf-8")
missing_file = RESULTS / "input_that_does_not_exist.jpg"
cases = [
    ("missing arguments", [], "InvalidArgument"),
    ("unknown flag", [left, right, output, "--version", "0", "--unknown"], "UnknownOption"),
    ("unsupported version", [left, right, output, "--version", "1"], "UnsupportedVersion"),
    ("non-positive runs", [left, right, output, "--version", "0", "--runs", "0"], "InvalidArgument"),
    ("missing input", [missing_file, right, output, "--version", "0"], "InputNotFound"),
    ("unreadable image", [bad_file, right, output, "--version", "0"], "ImageReadFailed"),
]
cli_rows = []
for label, arguments, expected in cases:
    result = execute([STITCH, *arguments], cwd=PROJECT, check=False)
    if result.returncode == 0 or f"[{expected}]" not in result.stderr:
        raise AssertionError("Unexpected CLI behavior: " + label)
    cli_rows.append({"case": label, "expected_error": expected, "exit_code": result.returncode})
display(pd.DataFrame(cli_rows))
CHECKS["cli_errors"] = True

<a id="results"></a>
## 3.5 Results and Profiling

### 3.5.1 Synthetic Homography Accuracy

Generator dùng ảnh gốc `raw/tuan/standard1_left.jpg`, hai trường hợp translation/projective và ba kích thước. H ground truth biến đổi B → A; ảnh B được lấy mẫu từ scene bằng bilinear. Không dùng correspondence thật từ H để thay bước tìm feature/match của pipeline.

Sai số đo trên lưới 20×20 của B, giữ các điểm rơi vào A theo **H ground truth**, không theo H ước lượng. RMSE/Max có đơn vị pixel của A; NRMSE chia RMSE cho đường chéo ảnh A. Ngưỡng nhóm chốt: **RMSE ≤ 0.5 px, Max ≤ 1 px trên cả sáu cặp**. Đây là ngưỡng thiết kế, không phải kết quả đo sẵn.

In [ ]:
from homography_common import load_truth
from homography_error import parse_estimate, evaluate

SYNTHETIC = RESULTS / "synthetic"
execute([sys.executable, "scripts/make_synthetic_pair.py", "--source", RAW_SYNTHETIC,
         "--out", SYNTHETIC], cwd=PROJECT)
synthetic_rows = []
for case in ("translation", "projective"):
    for size in SIZES:
        prefix = SYNTHETIC / case / size
        result = execute([STITCH, str(prefix) + "_left.png", str(prefix) + "_right.png",
                          str(prefix) + "_pano.png", "--version", "0", "--runs", "1", "--debug"], cwd=PROJECT)
        Path(str(prefix) + ".csv").write_text(result.stdout, encoding="utf-8")
        Path(str(prefix) + ".log").write_text(result.stderr, encoding="utf-8")
        parse_stitch_csv(result.stdout, runs=1)
        truth = load_truth(str(prefix) + "_truth.json")
        estimate = parse_estimate(result.stderr)
        metrics = evaluate(truth, estimate)
        passed = metrics["rmse_px"] <= 0.5 and metrics["max_px"] <= 1.0
        synthetic_rows.append({"case": case, "resolution": size, **metrics, "passed": passed})
        print("H_gt:", truth["H_gt"])
        print("H_est:", estimate.tolist())
synthetic_table = pd.DataFrame(synthetic_rows)
synthetic_table.to_csv(RESULTS / "synthetic_accuracy.csv", index=False)
display(synthetic_table)
CHECKS["synthetic_accuracy"] = len(synthetic_rows) == 6 and all(row["passed"] for row in synthetic_rows)
if not CHECKS["synthetic_accuracy"]:
    raise AssertionError("Synthetic accuracy failed; do not proceed with an accepted V0 baseline.")

### 3.5.2 Site Panoramas and Challenging Pair

Sinh panorama bằng cùng V0 và cấu hình đã chốt. Mỗi cặp dưới chạy một warm-up và một lần đo để tạo ảnh; **không dùng timing của bước này cho bảng benchmark chính thức**. PNG giữ chi tiết để crop, thumbnail JPEG chỉ dùng trình bày.

Với challenge ký túc xá, kiểm tra người đi bộ, xe nhòe và biên sáng/tối. Thành công thực thi không đồng nghĩa hết ghosting. Nhận xét chất lượng thuộc P3-T10 và phải dựa trên ảnh/crop vừa tạo.

In [ ]:
if PANORAMA_SIZE not in SIZES:
    raise ValueError("Unknown panorama resolution")
PANORAMAS = RESULTS / "panoramas"
PANORAMAS.mkdir()
panorama_rows = []
for site in SITES:
    prefix = PANORAMAS / f"{site}_{PANORAMA_SIZE}"
    result = execute([STITCH, DATASET / site / f"{PANORAMA_SIZE}_left.jpg",
                      DATASET / site / f"{PANORAMA_SIZE}_right.jpg", str(prefix) + ".png",
                      "--version", "0", "--runs", "1", "--debug"], cwd=PROJECT)
    Path(str(prefix) + ".csv").write_text(result.stdout, encoding="utf-8")
    Path(str(prefix) + ".log").write_text(result.stderr, encoding="utf-8")
    parse_stitch_csv(result.stdout, runs=1)
    with PILImage.open(str(prefix) + ".png") as image:
        dimensions = image.size
    panorama_rows.append({"site": site, "resolution": PANORAMA_SIZE,
                          "canvas_width": dimensions[0], "canvas_height": dimensions[1]})
    display(Markdown(f"**{site} · {PANORAMA_SIZE} · canvas {dimensions[0]} × {dimensions[1]}**"))
    display(DisplayImage(data=preview_image(str(prefix) + ".png")))
display(pd.DataFrame(panorama_rows))
CHECKS["panoramas"] = len(panorama_rows) == 4

### 3.5.3 Native-Resolution Inspection Crops

Cell dưới lấy các crop 512×512 gần phía trên, giữa và dưới bao giao hình học của hai ảnh, hiển thị **không resize**. Bao giao trục tọa độ không phải validity mask chính xác, nên cần kiểm tra vùng ảnh thật trong mỗi crop. Có thể thêm crop thủ công ở vị trí người/xe; không suy luận chất lượng từ thumbnail toàn cảnh.

In [ ]:
# Override a site with canvas-coordinate boxes (x0, y0, x1, y1) after the first inspection.
MANUAL_CROPS = {}  # Example: {"challenge": [(1200, 500, 1712, 1012)]}
crop_records = []
for site in SITES:
    prefix = PANORAMAS / f"{site}_{PANORAMA_SIZE}"
    width, height = SIZES[PANORAMA_SIZE]
    envelope = overlap_box(Path(str(prefix) + ".log").read_text(), width, height, width, height)
    x0, y0, x1, y1 = envelope
    boxes = MANUAL_CROPS.get(site, [])
    if not boxes:
        for fraction in (0.2, 0.5, 0.8):
            crop_w, crop_h = min(512, x1 - x0), min(512, y1 - y0)
            cx = max(x0, min((x0 + x1) // 2 - crop_w // 2, x1 - crop_w))
            cy = max(y0, min(int(y0 + fraction * (y1 - y0)) - crop_h // 2, y1 - crop_h))
            boxes.append((cx, cy, cx + crop_w, cy + crop_h))
    with PILImage.open(str(prefix) + ".png") as image:
        for index, box in enumerate(boxes, 1):
            a, b, c, d = box
            if not (0 <= a < c <= image.width and 0 <= b < d <= image.height):
                raise ValueError("Inspection crop leaves the canvas")
            crop_path = PANORAMAS / f"{site}_crop_{index}.png"
            image.crop(box).save(crop_path)
            display(Markdown(f"**{site} · crop {index} · {box} · 1:1 pixels**"))
            display(DisplayImage(filename=str(crop_path)))
            crop_records.append({"site": site, "box": list(box), "file": crop_path.name})
(RESULTS / "inspection_crops.json").write_text(json.dumps(crop_records, indent=2))
CHECKS["inspection_crops"] = len(crop_records) >= 4
CHECKS["quality_review"] = QUALITY_REVIEW_CONFIRMED

### 3.5.4 Official Timing Protocol

Benchmark chính: **site1/site2/site3 × small/medium/large**, một warm-up không đo và `--runs 5`. Đo bằng `std::chrono::high_resolution_clock` trong C++; không lấy thời gian Python/subprocess làm thời gian thuật toán. Đọc/ghi ảnh và debug ngoài vùng đo. Toàn bộ bảng phải đến từ cùng một phiên Colab T4, cùng nguồn và cờ build.

**P3-T04 integration — proposed, not yet confirmed:** script `scripts/run_benchmark.py` do Thành viết. Notebook truyền đối số qua `BENCHMARK_ARGS`. Đề xuất script nhận đường dẫn binary `STITCH`, dataset `DATASET`, số lần `RUNS` và output `BENCHMARK_ROOT`, rồi lưu nguyên stdout CSV vào `<output>/<site>/<size>.csv`. Khi script hoàn tất, chỉ cần điều chỉnh đối số hoặc adapter đường dẫn theo giao diện Thành xác nhận; không viết lại runner ở đây.

Nếu thiếu script/đối số, các bảng và biểu đồ bên dưới ghi trạng thái chờ. Chế độ nộp cuối `REQUIRE_COMPLETE_RUN=True` sẽ từ chối một phiên chưa có benchmark. Không dùng CSV từ phiên cũ để lấp chỗ trống.

In [ ]:
BENCHMARK_ROOT = RESULTS / "benchmark"
BENCHMARK_ROOT.mkdir()
benchmark_script = PROJECT / "scripts/run_benchmark.py"
BENCHMARK_EXECUTED = False
benchmark_average = benchmark_minimum = benchmark_runs = None
if not benchmark_script.is_file() or BENCHMARK_ARGS is None:
    print("ĐANG CHỜ P3-T04: chưa chạy benchmark chính thức; không có số liệu timing thay thế.")
    print("Binary:", STITCH, "Dataset:", DATASET, "Runs:", RUNS, "Output:", BENCHMARK_ROOT)
else:
    if not isinstance(BENCHMARK_ARGS, list) or not all(isinstance(x, str) for x in BENCHMARK_ARGS):
        raise TypeError("BENCHMARK_ARGS must be an argument list confirmed by Thanh.")
    # Substitutions are explicit argument values, never a shell command.
    substitutions = {"stitch": str(STITCH), "dataset": str(DATASET),
                     "runs": str(RUNS), "out": str(BENCHMARK_ROOT)}
    arguments = [argument.format(**substitutions) for argument in BENCHMARK_ARGS]
    result = execute([sys.executable, benchmark_script, *arguments], cwd=PROJECT, timeout=7200)
    (RESULTS / "benchmark_runner.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    average, minimum, measured = collect_benchmark_csv(BENCHMARK_ROOT, RUNS)
    benchmark_average, benchmark_minimum, benchmark_runs = map(pd.DataFrame, (average, minimum, measured))
    benchmark_average.to_csv(RESULTS / "timing_average.csv", index=False)
    benchmark_minimum.to_csv(RESULTS / "timing_minimum.csv", index=False)
    benchmark_runs.to_csv(RESULTS / "timing_runs.csv", index=False)
    BENCHMARK_EXECUTED = True
CHECKS["official_benchmark"] = BENCHMARK_EXECUTED

### 3.5.5 Per-Stage Timing and Hot Stages

Đơn vị milliseconds. S1–S3 bao gồm cả A và B. `Total_ms` được đo độc lập, có thể lớn hơn tổng sáu clock stage do overhead. Average và minimum tính từ từng lần đo, không tính trung bình của các minimum để mô tả một lần chạy thật.

In [ ]:
if BENCHMARK_EXECUTED:
    display(Markdown("**Average · milliseconds**"))
    display(benchmark_average.round(4))
    display(Markdown("**Minimum · milliseconds**"))
    display(benchmark_minimum.round(4))
    stage_columns = TIMING_COLUMNS[:-1]
    hot = benchmark_average[["site", "resolution"]].copy()
    hot["hot_stage"] = benchmark_average[stage_columns].idxmax(axis=1)
    hot["hot_stage_fraction"] = benchmark_average[stage_columns].max(axis=1) / benchmark_average["Total_ms"]
    display(hot)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    colors = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]
    for axis, site in zip(axes, SITES[:3]):
        table = benchmark_average[benchmark_average.site == site].set_index("resolution").loc[list(SIZES)]
        table[stage_columns].plot.bar(stacked=True, ax=axis, color=colors, rot=0, legend=False)
        axis.set(title=site, xlabel="Resolution", ylabel="Average time (ms)")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.savefig(RESULTS / "stage_timing.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    display(Markdown("Chưa có bảng timing/hot stage: chờ lần chạy P3-T04/T05 trong phiên này."))

### 3.5.6 Amdahl's Law

Với tập stage được nhóm chọn, $p = \sum T_{\mathrm{selected}} / T_{\mathrm{total}}$ và giới hạn lý tưởng là $S_{\infty}=1/(1-p)$. Tính riêng từng site/kích thước từ average của cùng các lần chạy. Mô hình này bỏ qua overhead truyền dữ liệu và GPU launch; không phải speedup đã đo trên GPU.

Nếu chưa có quyết định P3-T01 thì chưa công bố p cho kế hoạch của nhóm. Khi chọn cả sáu stage, phần còn lại có thể chỉ là overhead giữa các clock; giới hạn rất lớn hoặc vô hạn cần được giải thích trong ngữ cảnh đó.

In [ ]:
if BENCHMARK_EXECUTED and ACCELERATED_STAGES:
    if (len(set(ACCELERATED_STAGES)) != len(ACCELERATED_STAGES) or
            any(type(stage) is not int or stage not in range(1, 7) for stage in ACCELERATED_STAGES)):
        raise ValueError("Accelerated stages must be distinct integers from 1 to 6")
    selected_columns = [f"S{stage}_ms" for stage in ACCELERATED_STAGES]
    amdahl = benchmark_average[["site", "resolution"]].copy()
    if (benchmark_average.Total_ms <= 0).any():
        raise ValueError("Amdahl requires positive end-to-end measurements")
    amdahl["p"] = benchmark_average[selected_columns].sum(axis=1) / benchmark_average.Total_ms
    if ((amdahl.p < 0) | (amdahl.p > 1 + 1e-6)).any():
        raise ValueError("Stage fractions are inconsistent with total timing")
    amdahl["p"] = amdahl.p.clip(upper=1)
    amdahl["ideal_speedup_limit"] = amdahl.p.map(lambda p: float("inf") if p == 1 else 1 / (1 - p))
    display(amdahl)
    amdahl.to_csv(RESULTS / "amdahl.csv", index=False)
    CHECKS["amdahl_decision"] = True
else:
    print("Chờ benchmark và quyết định tập stage P3-T01; chưa tính Amdahl cho kế hoạch nhóm.")
    CHECKS["amdahl_decision"] = False

<a id="dataset"></a>
## 3.6 Dataset

### 3.6.1 Selected Image Pairs

Bộ cặp do nhóm chốt ngày 18/10. Tên địa điểm site1 vẫn cần Thành bổ sung; không tự suy đoán tên từ ảnh. Cặp chuẩn của Chương không thuộc bộ benchmark đã chốt; cặp ký túc xá của Chương là challenge.

| Site | Cặp | Địa điểm | Vai trò |
|:--|:--|:--|:--|
| site1 | `thanh-standard1` | Chờ Thành xác nhận tên | Chuẩn |
| site2 | `tuan-standard2` | Ẩn Coffee, trong nhà | Chuẩn |
| site3 | `de-standard2` | Mai An Coffee, ngoài trời | Chuẩn |
| challenge | `chuong-challenge1` | Ký túc xá, ngoài trời | Cảnh đêm, chênh sáng và chuyển động |

### 3.6.2 Preparation and Access

Ảnh gốc được xoay theo EXIF, hai ảnh trong cặp cắt cùng khung 16:9, resize Lanczos ra 1280×720, 1920×1080 và 3840×2160; lưu JPEG RGB quality 95, bỏ EXIF. Manifest lưu nguồn, crop box và SHA-256. Cell 0.4 kiểm tra 24 ảnh của bản thực sự dùng trong phiên này.

- [Prepared dataset](https://drive.google.com/drive/folders/1HOvnly1k1QHOsV17qLzSAG4ihszNzCqO?usp=sharing)
- [Original photographs](https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing)

Metadata và quy trình đầy đủ nằm trong `docs/dataset.md`; mục 3.6 của nhóm được ghép từ `docs/report/3.6-dataset.md` ở P4.

<a id="gpu-plan"></a>
## 3.7 GPU Execution Plan

### 3.7.1 Preliminary Thread Mapping

Bảng sau là định hướng từ thiết kế hiện có; chưa phải kết quả CUDA hoặc quyết định tối ưu sau profiling. Tuấn tổng hợp bản chính thức từ mục 6 của sáu tài liệu stage trong P3-T03.

| Stage | Mapping V1 dự kiến | Rủi ro chính |
|:--|:--|:--|
| S1 | Thread/pixel cho grayscale và từng pass Gaussian | Halo, border, global-memory traffic |
| S2 | Thread/pixel cho FAST và NMS; bước chọn top-N riêng | Tie-break, compaction/selection, mật độ keypoint |
| S3 | Song song theo keypoint và các test descriptor | Gather, packing bit, giữ thứ tự keypoint |
| S4 | Song song các cặp descriptor, reduction lấy 2-NN | Traffic, tie-break, ratio test nhất quán |
| S5 | Hypothesis/residual song song; reduction score | Seed/sample prefix, adaptive stopping, precision |
| S6 | Thread/pixel canvas, inverse mapping và composite | Projective gather, canvas allocation, precision/biên |

### 3.7.2 Optimization Candidates and Evidence Requirements

Các hướng khảo sát gồm giảm buffer trung gian/fusion, điều chỉnh block layout và tận dụng reuse khi profiling chứng minh có lợi. Với S6, shared memory không mặc nhiên hiệu quả vì projective gather không luôn khớp tile. Hardware texture bilinear không mặc nhiên đồng nhất với bilinear tự tính; không đổi phương pháp hoặc tolerance âm thầm để tăng tốc.

V1 phải giữ cùng phương pháp và tham số V0. V2/V3 chỉ chốt sau profiling V1. Kế hoạch phải mô tả cả bbox/inverse, reduction, metadata và host/device transfer; không chỉ gọi vòng lặp lớn là toàn bộ stage. Phần lý do chọn stage phải được cập nhật từ bảng timing và Amdahl ở trên.

**Chờ P3-T01/P3-T03:** quyết định tập stage, mapping đã thống nhất và ưu tiên theo số đo. Phần này chưa được coi là kế hoạch cuối cùng.

<a id="references"></a>
## References

Nguồn dưới đây kế thừa tài liệu thiết kế đã có; notebook này không thực hiện một đợt kiểm chứng thư mục tài liệu mới. Nguồn gốc thuật toán và nguồn code cần được phân biệt; các số đo của nhóm chỉ lấy từ cell output.

| Nguồn | Phần sử dụng |
|:--|:--|
| Đề giữa kỳ và cuối kỳ CSC14120 trong `docs/specs/` | Cấu trúc 3.1–3.7, baseline, đo và nộp bài |
| `docs/midterm-plan.md`, `docs/task-board.md`, `docs/interface.md` | Phân công, dữ liệu, timing, ownership và nghiệm thu |
| `docs/stages/s1-preprocessing.md` đến `s6-warp-blend.md` | Thiết kế và bằng chứng đã ghi của từng stage |
| Rosten & Drummond (2006), [FAST paper](https://www.edwardrosten.com/work/rosten_2006_machine.pdf) | Nguồn phương pháp S2, theo tài liệu S2 |
| Calonder et al. (2010), [BRIEF paper](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf) | Nguồn descriptor S3, theo tài liệu S3 |
| Brown & Lowe (2007), [Automatic Panoramic Image Stitching](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf) | Bối cảnh matching/homography/blending, theo S4–S6 |
| Szeliski, [Image Alignment and Stitching: A Tutorial](https://doi.org/10.1561/0600000009) | Bối cảnh resampling/compositing, theo tài liệu S6 |
| `docs/report/references.md` và comment đầu các file nguồn | Danh mục nguồn chi tiết và ghi nhận code ngoài |

Notebook dùng hỗ trợ AI để tổ chức nội dung và khung thực thi. Thành viên nhóm chịu trách nhiệm kiểm tra mã, chạy thực nghiệm và xác nhận các phân tích trước khi nộp. Không ghi nhận một nguồn là đã được con người đọc chỉ vì xuất hiện trong bảng này.

<a id="checklist"></a>
## Reproducibility Checklist

Cell cuối tổng hợp trạng thái phiên hiện tại và lưu manifest bằng chứng. **READY** chỉ xuất hiện khi mọi kiểm tra và xác nhận đều đạt; chưa chạy benchmark hoặc chưa review chất lượng sẽ giữ trạng thái **INCOMPLETE**.

Trước khi nộp: điền địa điểm site1, ghép bản nháp P3 của nhóm, xác nhận chất lượng và kế hoạch GPU; bật `REQUIRE_COMPLETE_RUN=True`, chạy Run all trên T4 mới rồi lưu notebook có output. Bản làm việc đưa lên Git giữ `execution_count=null`, `outputs=[]`.

In [ ]:
CHECKS["source_unchanged"] = source_manifest(PROJECT) == SOURCE_HASHES
CHECKS["quality_review"] = QUALITY_REVIEW_CONFIRMED
CHECKS["final_report"] = FINAL_REPORT_CONFIRMED
display(pd.DataFrame([{"check": name, "passed": bool(value)} for name, value in CHECKS.items()]))
ready = all(CHECKS.values())
receipt = {"session_id": SESSION_ID, "status": "READY" if ready else "INCOMPLETE",
           "checks": CHECKS, "accelerated_stages": ACCELERATED_STAGES,
           "raw_synthetic_sha256": raw_digest,
           "evidence_files": sorted(str(path.relative_to(RESULTS)) for path in RESULTS.rglob("*") if path.is_file())}
(RESULTS / "run_receipt.json").write_text(json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8")
print("Run status:", receipt["status"])
print("Evidence directory:", RESULTS)
if REQUIRE_COMPLETE_RUN and not ready:
    raise RuntimeError("Final submission gate failed: complete all pending evidence and report checks.")